# 04 — Synthetic ground-truth topology recovery

This notebook measures end-to-end Yamada-invariant recovery from a known spatial graph through

$$G_{\rm true}\to G_\lambda\to V_{\lambda,N}\to\widehat G\to\Upsilon(\widehat G).$$

The deformation $G_{\rm true}\to G_\lambda$ is restricted to orientation-preserving invertible affine maps $F(x)=Mx+b$ with $\det M>0$. Because $GL^+(3,\mathbb R)$ is path-connected, these maps are ambient-isotopic to the identity and cannot change the spatial-graph topology. Arbitrary Gaussian geometric noise is therefore not used.

Success means $\Upsilon(\widehat G)=\Upsilon(G_{\rm true})$. This is invariant recovery, not proof of graph equivalence, because Yamada is not complete. Continuously over-thick tubes are excluded by a conservative admissibility guard; failures after admissible construction are genuine voxelization/skeletonization/extraction failures.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize
from knotted_graph.core import simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

A = sp.Symbol("A")
QUICK_MODE = True  # set False for the larger sweep
BOUND = 1.35
SMOOTH_EPSILON = 0.0
PROJECTION_SAMPLES = 8
if QUICK_MODE:
    RESOLUTIONS = [56]
    TUBE_RADII_VOX = [1, 2]
    TRANSFORMS = ["identity", "rotate", "affine"]
else:
    RESOLUTIONS = [40, 48, 56, 64, 72, 80, 96, 112, 128]
    TUBE_RADII_VOX = [1, 2, 3]
    TRANSFORMS = ["identity", "rotate", "affine", "affine_2", "affine_3", "affine_4"]

## Ground-truth suite

The quick suite uses an unknot, a trivalent $\Theta_3$, and an unknotted spatial $K_4$. A degree-4 bouquet is intentionally excluded from the headline suite: a thickened genus-2 handlebody can admit different spines (e.g. bouquet-like and theta-like), so the original high-valence spine is not a unique medial-skeleton ground truth.

In [ ]:
@dataclass
class Case:
    name: str
    graph: nx.MultiGraph
    safe_radius_cap: float

def embedded(nodes, edges):
    G = nx.MultiGraph()
    for n, p in nodes.items(): G.add_node(n, pos=np.asarray(p, float))
    for u, v, pts in edges: G.add_edge(u, v, pts=np.asarray(pts, float))
    return G

def unknot(n=360):
    t=np.linspace(0,2*np.pi,n); pts=np.c_[.62*np.cos(t),.62*np.sin(t),np.zeros_like(t)]
    return Case("unknot",embedded({0:pts[0]},[(0,0,pts)]),.20)

def theta3(n=300):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    curves=[np.c_[x,a*np.sin(np.pi*t),np.zeros_like(t)] for a in (-.58,0,.58)]
    return Case("theta3",embedded({"u":curves[0][0],"v":curves[0][-1]},[("u","v",p) for p in curves]),.10)

def k4(n=220):
    xyz=np.array([[.65,.65,.65],[.65,-.65,-.65],[-.65,.65,-.65],[-.65,-.65,.65]])*.72
    base=nx.complete_graph(4)
    return Case("K4",embedded({i:xyz[i] for i in base},[(u,v,np.linspace(xyz[u],xyz[v],n)) for u,v in base.edges()]),.075)

CASES=[unknot(),theta3(),k4()]
[(c.name,c.graph.number_of_nodes(),c.graph.number_of_edges()) for c in CASES]

In [ ]:
def Rxyz(ax,ay,az):
    ax,ay,az=np.deg2rad([ax,ay,az])
    Rx=np.array([[1,0,0],[0,np.cos(ax),-np.sin(ax)],[0,np.sin(ax),np.cos(ax)]])
    Ry=np.array([[np.cos(ay),0,np.sin(ay)],[0,1,0],[-np.sin(ay),0,np.cos(ay)]])
    Rz=np.array([[np.cos(az),-np.sin(az),0],[np.sin(az),np.cos(az),0],[0,0,1]])
    return Rz@Ry@Rx

def affine(name):
    if name=="identity": M,b=np.eye(3),np.zeros(3)
    elif name=="rotate": M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    elif name=="affine":
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    else:
        seed={"affine_2":20260819,"affine_3":20260820,"affine_4":20260821}[name]
        rng=np.random.default_rng(seed); R=Rxyz(*rng.uniform(-40,40,3)); S=np.diag(rng.uniform(.88,1.12,3)); H=np.eye(3)+rng.uniform(-.10,.10,(3,3))
        if np.linalg.det(H)<0: H[0]*=-1
        M,b=R@S@H,rng.uniform(-.04,.04,3)
    assert np.linalg.det(M)>0
    return M,b

def deform(G,name):
    M,b=affine(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d["pos"])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d["pts"])@M.T+b)
    return H

[(name,float(np.linalg.det(affine(name)[0]))) for name in TRANSFORMS]

In [ ]:
def interior_separation(G,trim=.15):
    E=[(u,v,np.asarray(d["pts"],float)) for u,v,k,d in G.edges(keys=True,data=True)]
    if len(E)<2: return np.inf
    best=np.inf
    for i,(u1,v1,p1) in enumerate(E):
        for u2,v2,p2 in E[i+1:]:
            if {u1,v1}&{u2,v2}:
                a1=max(1,int(trim*len(p1))); a2=max(1,int(trim*len(p2)))
                q1=p1[a1:-a1] if 2*a1<len(p1) else p1; q2=p2[a2:-a2] if 2*a2<len(p2) else p2
            else: q1,q2=p1,p2
            for s in range(0,len(q1),128):
                d2=np.sum((q1[s:s+128,None,:]-q2[None,:,:])**2,axis=-1); best=min(best,float(np.sqrt(d2.min())))
    return best

def admissible(case,G,N,rvox):
    dx=2*BOUND/(N-1); rw=rvox*dx; sep=interior_separation(G); sep_cap=np.inf if not np.isfinite(sep) else .40*sep
    limit=min(case.safe_radius_cap,sep_cap)
    return rw<=limit,rw,sep,limit

def resample(pts,step):
    pts=np.asarray(pts,float); out=[]
    for p,q in zip(pts[:-1],pts[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1); out.append(np.linspace(p,q,n,endpoint=False))
    out.append(pts[-1:]); return np.vstack(out)

def voxelize(G,N,rvox):
    V=np.zeros((N,N,N),bool); dx=2*BOUND/(N-1)
    for u,v,k,d in G.edges(keys=True,data=True):
        P=resample(d["pts"],dx/3); I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int); I=np.clip(I,0,N-1); V[I[:,0],I[:,1],I[:,2]]=True
    return dilation(V,footprint=ball(int(rvox)))

def world_graph(G,N):
    H=nx.MultiGraph(G); dx=2*BOUND/(N-1); o=np.array([-BOUND]*3,float)
    for n,d in H.nodes(data=True): d["pos"]=o+dx*np.asarray(d["pos"],float)
    for u,v,k,d in H.edges(keys=True,data=True): d["pts"]=o+dx*np.asarray(d["pts"],float)
    return H

def extract(V,N):
    S=skeletonize(V,method="lee"); raw=skeleton_image_to_graph(S); G=simplify_edges(world_graph(raw,N))
    if SMOOTH_EPSILON>0: G=smooth_edges(G,epsilon=SMOOTH_EPSILON)
    return S,raw,G

def yamada(G):
    r=compute_yamada_polynomial(G,A,rotation_angles=None,num_rotation_samples=PROJECTION_SAMPLES,normalize=False,n_jobs=1,method="recursive",return_result=True)
    return sp.expand(r.polynomial),int(r.projection.num_crossings)

def same(a,b): return sp.simplify(sp.together(sp.expand(a-b)))==0

In [ ]:
targets={}
for c in CASES:
    targets[c.name],cross=yamada(c.graph); print(c.name,"target crossings=",cross,"Yamada=",targets[c.name])

records=[]
for c in CASES:
    for tname in TRANSFORMS:
        Gt=deform(c.graph,tname)
        for N in RESOLUTIONS:
            for rv in TUBE_RADII_VOX:
                ok_geom,rw,sep,limit=admissible(c,Gt,N,rv)
                rec=dict(case=c.name,transform=tname,resolution=N,radius_vox=rv,radius_world=rw,separation=sep,continuous_limit=limit,admissible=bool(ok_geom),success=None,final_V=None,final_E=None,error=None)
                if ok_geom:
                    try:
                        S,raw,Ghat=extract(voxelize(Gt,N,rv),N); yp,cross=yamada(Ghat)
                        rec.update(success=bool(same(yp,targets[c.name])),final_V=Ghat.number_of_nodes(),final_E=Ghat.number_of_edges(),crossings=cross,recovered_polynomial=str(yp))
                    except Exception as e:
                        rec.update(success=False,error=f"{type(e).__name__}: {e}")
                records.append(rec)
                mark="SKIP" if not ok_geom else ("PASS" if rec["success"] else "FAIL")
                print(f"{mark:4s} {c.name:8s} {tname:8s} N={N:3d} r={rv} V/E={rec['final_V']}/{rec['final_E']}")

valid=[r for r in records if r["admissible"]]; passed=sum(bool(r["success"]) for r in valid)
print(f"\nHeadline Yamada recovery: {passed}/{len(valid)} = {100*passed/len(valid):.2f}%")
print("Excluded by continuous-thickness guard:",len(records)-len(valid))
for r in valid:
    if not r["success"]: print("FAILURE:",r["case"],r["transform"],"N=",r["resolution"],"r=",r["radius_vox"],"V/E=",r["final_V"],r["final_E"],r["error"] or "Yamada mismatch")

In [ ]:
res=sorted({r["resolution"] for r in records}); rad=sorted({r["radius_vox"] for r in records}); H=np.full((len(rad),len(res)),np.nan)
for i,rv in enumerate(rad):
    for j,N in enumerate(res):
        g=[r for r in records if r["admissible"] and r["radius_vox"]==rv and r["resolution"]==N]
        if g: H[i,j]=np.mean([bool(r["success"]) for r in g])
fig,ax=plt.subplots(figsize=(max(5,.8*len(res)),3.2)); im=ax.imshow(H,vmin=0,vmax=1,aspect="auto",origin="lower")
ax.set_xticks(range(len(res)),res); ax.set_yticks(range(len(rad)),rad); ax.set_xlabel("voxel resolution N"); ax.set_ylabel("tube radius [voxels]"); ax.set_title("Yamada invariant recovery rate"); fig.colorbar(im,ax=ax,label="recovery fraction"); plt.show()

## Interpretation / large run

`admissible=False` is excluded because the requested continuous tube is too thick for the conservative geometry guard. `admissible=True, success=False` is a meaningful end-to-end failure after a topology-preserving centerline deformation. The resulting resolution/thickness boundary is therefore a practical validity regime for the pipeline.

Set `QUICK_MODE=False` for the larger deterministic sweep. The next scientifically clean expansion is more **bridgeless trivalent spatial graphs** and knot embeddings; avoid treating arbitrary high-valence handlebody spines as unique graph ground truths without an extra convention selecting a spine.